## 使用命名实体识别（NER）丰富文本

`Named Entity Recognition`（NER，命名实体识别）是一项 `Natural Language Processing`（自然语言处理）任务，用于识别命名实体（NE）并将其归入预定义的语义类别（例如人物、组织、地点、事件、时间表达式和数量）。通过将原始文本转换为结构化信息，NER 让数据更具可操作性，有助于完成信息抽取、数据聚合、分析以及社交媒体监测等任务。

本 notebook 演示如何结合 [chat completion](https://platform.openai.com/docs/api-reference/chat) 和 [函数调用（functions-calling）](https://platform.openai.com/docs/guides/gpt/function-calling) 执行 NER，从而为文本添加指向 Wikipedia 等知识库的链接：

**文本：**

*In Germany, in 1440, goldsmith Johannes Gutenberg invented the movable-type printing press. His work led to an information revolution and the unprecedented mass-spread of literature throughout Europe. Modelled on the design of the existing screw presses, a single Renaissance movable-type printing press could produce up to 3,600 pages per workday.*

**添加了 Wikipedia 链接的文本：**

*In [Germany](https://en.wikipedia.org/wiki/Germany), in 1440, goldsmith [Johannes Gutenberg]() invented the [movable-type printing press](https://en.wikipedia.org/wiki/Movable_Type). His work led to an [information revolution](https://en.wikipedia.org/wiki/Information_revolution) and the unprecedented mass-spread of literature throughout [Europe](https://en.wikipedia.org/wiki/Europe). Modelled on the design of the existing screw presses, a single [Renaissance](https://en.wikipedia.org/wiki/Renaissance) [movable-type printing press](https://en.wikipedia.org/wiki/Movable_Type) could produce up to 3,600 pages per workday.*

**推理成本：** 本 notebook 还说明了如何估算 OpenAI API 的成本。

### 1. 设置

#### 1.1 安装/升级 Python 软件包

In [1]:
%pip install --upgrade openai --quiet
%pip install --upgrade nlpia2-wikipedia --quiet
%pip install --upgrade tenacity --quiet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


#### 1.2 加载软件包和 OPENAI_API_KEY

你可以在 OpenAI 网页界面中生成 API 密钥。详情请参阅 https://platform.openai.com/account/api-keys 。

本 notebook 适用于最新的 OpenAI 模型 `gpt-3.5-turbo-0613` 和 `gpt-4-0613`。

In [2]:
import json
import logging
import os

import openai
import wikipedia

from typing import Optional
from IPython.display import display, Markdown
from tenacity import retry, wait_random_exponential, stop_after_attempt

logging.basicConfig(level=logging.INFO, format=' %(asctime)s - %(levelname)s - %(message)s')

OPENAI_MODEL = 'gpt-3.5-turbo-0613'

client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))

### 2. 定义需要识别的 NER 标签

我们定义了一组标准的 NER 标签，以展示广泛的用例。不过，对于用知识库链接丰富文本这一特定任务，实际上只需要其中的一部分。

In [3]:
labels = [
    "person",      # people, including fictional characters
    "fac",         # buildings, airports, highways, bridges
    "org",         # organizations, companies, agencies, institutions
    "gpe",         # geopolitical entities like countries, cities, states
    "loc",         # non-gpe locations
    "product",     # vehicles, foods, appareal, appliances, software, toys 
    "event",       # named sports, scientific milestones, historical events
    "work_of_art", # titles of books, songs, movies
    "law",         # named laws, acts, or legislations
    "language",    # any named language
    "date",        # absolute or relative dates or periods
    "time",        # time units smaller than a day
    "percent",     # percentage (e.g., "twenty percent", "18%")
    "money",       # monetary values, including unit
    "quantity",    # measurements, e.g., weight or distance
]

### 3. 准备消息

[chat completions API](https://platform.openai.com/docs/guides/gpt/chat-completions-api) 接收一个消息列表作为输入，并输出一条由模型生成的消息。虽然对话格式主要是为多轮对话设计的，但它同样能高效处理没有先前对话的单轮任务。在这里，我们将为 system、assistant 和 user 三种角色分别指定消息。

#### 3.1 系统消息

`system message`（提示词）通过定义助手期望的角色设定和任务来设定其行为。我们还会明确列出要识别的实体标签集合。

虽然可以指示模型如何格式化回复，但需要注意的是，`gpt-3.5-turbo-0613` 和 `gpt-4-0613` 都经过了微调，能够判断何时应调用函数，并按照函数签名返回 `JSON` 格式的回复。这一能力简化了我们的提示词，使我们能够直接从模型获得结构化数据。

In [4]:
def system_message(labels):
    return f"""
You are an expert in Natural Language Processing. Your task is to identify common Named Entities (NER) in a given text.
The possible common Named Entities (NER) types are exclusively: ({", ".join(labels)})."""

#### 3.2 助手消息

`Assistant messages` 通常用于存储助手之前的回复。不过，正如我们的场景所示，也可以专门编写它们来提供期望行为的示例。虽然 OpenAI 能够执行 `zero-shot` 命名实体识别，但我们发现 `one-shot` 方法能产生更精确的结果。

In [5]:
def assisstant_message():
    return f"""
EXAMPLE:
    Text: 'In Germany, in 1440, goldsmith Johannes Gutenberg invented the movable-type printing press. His work led to an information revolution and the unprecedented mass-spread / 
    of literature throughout Europe. Modelled on the design of the existing screw presses, a single Renaissance movable-type printing press could produce up to 3,600 pages per workday.'
    {{
        "gpe": ["Germany", "Europe"],
        "date": ["1440"],
        "person": ["Johannes Gutenberg"],
        "product": ["movable-type printing press"],
        "event": ["Renaissance"],
        "quantity": ["3,600 pages"],
        "time": ["workday"]
    }}
--"""

#### 3.3 用户消息

`user message` 为助手的任务提供具体文本：

In [6]:
def user_message(text):
    return f"""
TASK:
    Text: {text}
"""

### 4. OpenAI 函数（及工具函数）

在 OpenAI API 调用中，我们可以向 `gpt-3.5-turbo-0613` 和 `gpt-4-0613` 描述 `functions`，让模型智能地选择输出一个 `JSON` 对象，其中包含调用这些 `functions` 所需的参数。需要注意的是，[chat completions API](https://platform.openai.com/docs/guides/gpt/chat-completions-api) 并不会真正执行 `function`。它只提供 `JSON` 输出，我们随后可以在代码中用它来调用 `function`。更多详情，请参阅 [OpenAI 函数调用指南](https://platform.openai.com/docs/guides/function-calling)。

我们的函数 `enrich_entities(text, label_entities)` 接收一段文本，以及一个包含已识别标签和实体的字典作为参数。然后，它会将识别出的实体与对应的 Wikipedia 文章链接关联起来。

In [7]:
@retry(wait=wait_random_exponential(min=1, max=10), stop=stop_after_attempt(5))
def find_link(entity: str) -> Optional[str]:
    """
    Finds a Wikipedia link for a given entity.
    """
    try:
        titles = wikipedia.search(entity)
        if titles:
            # naively consider the first result as the best
            page = wikipedia.page(titles[0])
            return page.url
    except (wikipedia.exceptions.WikipediaException) as ex:
        logging.error(f'Error occurred while searching for Wikipedia link for entity {entity}: {str(ex)}')

    return None

In [8]:
def find_all_links(label_entities:dict) -> dict:
    """ 
    Finds all Wikipedia links for the dictionary entities in the whitelist label list.
    """
    whitelist = ['event', 'gpe', 'org', 'person', 'product', 'work_of_art']
    
    return {e: find_link(e) for label, entities in label_entities.items() 
                            for e in entities
                            if label in whitelist}

In [9]:
def enrich_entities(text: str, label_entities: dict) -> str:
    """
    Enriches text with knowledge base links.
    """
    entity_link_dict = find_all_links(label_entities)
    logging.info(f"entity_link_dict: {entity_link_dict}")
    
    for entity, link in entity_link_dict.items():
        text = text.replace(entity, f"[{entity}]({link})")

    return text

### 4. ChatCompletion

如前所述，`gpt-3.5-turbo-0613` 和 `gpt-4-0613` 经过微调，能够检测何时应调用 `function`。此外，它们还能生成符合 `function` 签名的 `JSON` 回复。我们遵循的步骤如下：

1. 定义我们的 `function` 及其对应的 `JSON` Schema。
2. 使用 `messages`、`tools` 和 `tool_choice` 参数调用模型。
3. 将输出转换为 `JSON` 对象，然后使用模型提供的 `arguments` 调用 `function`。

在实践中，你可能希望将 `function` 的响应作为新消息追加，再次调用模型，让模型把结果总结给用户。不过，就我们的目的而言，这一步并不需要。

*请注意，在真实场景中，强烈建议在执行操作之前加入用户确认流程。*

#### 4.1 定义函数和 JSON schema

由于我们希望模型输出一个由标签和识别出的实体组成的字典：

```python
{   
    "gpe": ["Germany", "Europe"],   
    "date": ["1440"],   
    "person": ["Johannes Gutenberg"],   
    "product": ["movable-type printing press"],   
    "event": ["Renaissance"],   
    "quantity": ["3,600 pages"],   
    "time": ["workday"]   
}   
```
我们需要定义相应的 `JSON` schema，并将其传给 `tools` 参数：

In [10]:
def generate_functions(labels: dict) -> list:
    return [
        {   
            "type": "function",
            "function": {
                "name": "enrich_entities",
                "description": "Enrich Text with Knowledge Base Links",
                "parameters": {
                    "type": "object",
                        "properties": {
                            "r'^(?:' + '|'.join({labels}) + ')$'": 
                            {
                                "type": "array",
                                "items": {
                                    "type": "string"
                                }
                            }
                        },
                        "additionalProperties": False
                },
            }
        }
    ]

#### 4.2 Chat Completion

现在我们来调用模型。需要注意的是，我们通过将 `tool_choice` 参数设置为 `{"type": "function", "function" : {"name": "enrich_entities"}}`，指示 API 使用特定的函数。

In [11]:
@retry(wait=wait_random_exponential(min=1, max=10), stop=stop_after_attempt(5))
def run_openai_task(labels, text):
    messages = [
          {"role": "system", "content": system_message(labels=labels)},
          {"role": "assistant", "content": assisstant_message()},
          {"role": "user", "content": user_message(text=text)}
      ]

    # TODO: functions and function_call are deprecated, need to be updated
    # See: https://platform.openai.com/docs/api-reference/chat/create#chat-create-tools
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo-0613",
        messages=messages,
        tools=generate_functions(labels),
        tool_choice={"type": "function", "function" : {"name": "enrich_entities"}}, 
        temperature=0,
        frequency_penalty=0,
        presence_penalty=0,
    )

    response_message = response.choices[0].message
    
    available_functions = {"enrich_entities": enrich_entities}  
    function_name = response_message.tool_calls[0].function.name
    
    function_to_call = available_functions[function_name]
    logging.info(f"function_to_call: {function_to_call}")

    function_args = json.loads(response_message.tool_calls[0].function.arguments)
    logging.info(f"function_args: {function_args}")

    function_response = function_to_call(text, function_args)

    return {"model_response": response, 
            "function_response": function_response}

### 5. 用 Wikipedia 链接丰富文本

#### 5.1 运行 OpenAI 任务

In [12]:
text = """The Beatles were an English rock band formed in Liverpool in 1960, comprising John Lennon, Paul McCartney, George Harrison, and Ringo Starr."""
result = run_openai_task(labels, text)

 2023-10-20 18:05:51,729 - INFO - function_to_call: <function enrich_entities at 0x0000021D30C462A0>
 2023-10-20 18:05:51,730 - INFO - function_args: {'person': ['John Lennon', 'Paul McCartney', 'George Harrison', 'Ringo Starr'], 'org': ['The Beatles'], 'gpe': ['Liverpool'], 'date': ['1960']}
 2023-10-20 18:06:09,858 - INFO - entity_link_dict: {'John Lennon': 'https://en.wikipedia.org/wiki/John_Lennon', 'Paul McCartney': 'https://en.wikipedia.org/wiki/Paul_McCartney', 'George Harrison': 'https://en.wikipedia.org/wiki/George_Harrison', 'Ringo Starr': 'https://en.wikipedia.org/wiki/Ringo_Starr', 'The Beatles': 'https://en.wikipedia.org/wiki/The_Beatles', 'Liverpool': 'https://en.wikipedia.org/wiki/Liverpool'}


#### 5.2 函数响应

In [13]:
display(Markdown(f"""**Text:** {text}   
                     **Enriched_Text:** {result['function_response']}"""))

**Text:** The Beatles were an English rock band formed in Liverpool in 1960, comprising John Lennon, Paul McCartney, George Harrison, and Ringo Starr.   
                     **Enriched_Text:** [The Beatles](https://en.wikipedia.org/wiki/The_Beatles) were an English rock band formed in [Liverpool](https://en.wikipedia.org/wiki/Liverpool) in 1960, comprising [John Lennon](https://en.wikipedia.org/wiki/John_Lennon), [Paul McCartney](https://en.wikipedia.org/wiki/Paul_McCartney), [George Harrison](https://en.wikipedia.org/wiki/George_Harrison), and [Ringo Starr](https://en.wikipedia.org/wiki/Ringo_Starr).

#### 5.3 Token 用量

要估算推理成本，我们可以解析响应中的 "usage" 字段。各模型详细的 token 价格请参阅 [OpenAI 定价指南](https://openai.com/pricing)：

In [14]:
# estimate inference cost assuming gpt-3.5-turbo (4K context)
i_tokens  = result["model_response"].usage.prompt_tokens 
o_tokens = result["model_response"].usage.completion_tokens 

i_cost = (i_tokens / 1000) * 0.0015
o_cost = (o_tokens / 1000) * 0.002

print(f"""Token Usage
    Prompt: {i_tokens} tokens
    Completion: {o_tokens} tokens
    Cost estimation: ${round(i_cost + o_cost, 5)}""")

Token Usage
    Prompt: 331 tokens
    Completion: 47 tokens
    Cost estimation: $0.00059
